# Trabalho Prático — Mineração de Dados 2026-2
## Pitch Intelligence: Pipeline de Inteligência Esportiva

**Universidade de Marília (UNIMAR)** | Prazo: 13/06/2026

Este notebook implementa as 5 etapas do trabalho:
1. Construção da base (FBref + Transfermarkt)
2. Pré-processamento e EDA
3. Clusterização (K-Means)
4. Classificação (Titular Regular vs Rotação/Reserva)
5. Insumos para recomendação estratégica

> **Colab:** execute as células em ordem. Na primeira execução, faça upload dos arquivos do repositório ou clone o repo.


In [2]:
# @title 0. Setup — dependências e caminhos
# %pip instala no kernel ativo do notebook (use isto no Cursor/VS Code, não !pip)
%pip install -q numpy pandas matplotlib seaborn scikit-learn beautifulsoup4 requests

import os
import re
import time
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from bs4 import BeautifulSoup

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BASE_DIR = Path(".")
if IN_COLAB and not (BASE_DIR / "fbref_big5_brasileirao_2526_V2.csv").exists():
    print("Arquivos não encontrados. Opções:")
    print("  1) Clone: !git clone https://github.com/SEU_USUARIO/TrabalhoPos.git && cd TrabalhoPos")
    print("  2) Upload manual dos CSVs e pasta data_html/")

DATA_DIR = BASE_DIR / "dados"
DATA_DIR.mkdir(exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
print(f"Ambiente: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Diretório base: {BASE_DIR.resolve()}")



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Users\jesus\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'numpy'

### Upload de arquivos (Google Colab)

Se os CSVs não estiverem no diretório, execute a célula abaixo para fazer upload:
- `fbref_big5_brasileirao_2526_V2.csv`
- `fbref_big5_brasileirao_2526.csv`
- Pasta `data_html/` (opcional, para validação de goleiros)


In [ ]:
# @title Upload de arquivos (somente Colab)
if IN_COLAB and not (BASE_DIR / "fbref_big5_brasileirao_2526_V2.csv").exists():
    from google.colab import files
    print("Selecione os arquivos CSV do repositório:")
    uploaded = files.upload()
    for nome in uploaded:
        print(f"  Recebido: {nome}")
else:
    print("Arquivos já presentes ou ambiente local — upload não necessário.")


In [ ]:
# @title Funções utilitárias

def normalizar_nome(nome):
    """Remove acentos e pontuação para facilitar o merge."""
    if pd.isna(nome) or nome == "":
        return ""
    nome = unicodedata.normalize("NFKD", str(nome))
    nome = "".join(c for c in nome if not unicodedata.combining(c))
    nome = re.sub(r"[^a-z0-9]", "", nome.lower())
    return nome


def parse_idade(age):
    if pd.isna(age):
        return np.nan
    s = str(age)
    if "-" in s:
        return int(s.split("-")[0])
    return int(float(s))


def simplificar_posicao(pos):
    """Regra: primeira sigla em combinações (ex. DF,MF vira DF)."""
    if pd.isna(pos):
        return "MF"
    primeira = str(pos).split(",")[0].strip()
    for sigla in ["GK", "DF", "MF", "FW"]:
        if sigla in primeira:
            return sigla
    return primeira[:2] if len(primeira) >= 2 else "MF"


def parse_valor_mercado(valor_str):
    """Converte strings do Transfermarkt para float em euros."""
    if pd.isna(valor_str) or str(valor_str).strip() == "":
        return np.nan
    s = str(valor_str).replace("€", "").strip()
    multiplier = 1.0
    if re.search(r"[Mm]i", s):
        multiplier = 1_000_000
        s = re.sub(r"[Mm]i\.?", "", s, flags=re.IGNORECASE)
    elif re.search(r"[Mm]il|[Tt]h", s):
        multiplier = 1_000
        s = re.sub(r"[Mm]il\.?|[Tt]h\.?", "", s, flags=re.IGNORECASE)
    s = s.strip().replace(" ", "")
    if "," in s and "." in s:
        if s.rfind(",") > s.rfind("."):
            s = s.replace(".", "").replace(",", ".")
        else:
            s = s.replace(",", "")
    elif "," in s:
        s = s.replace(".", "").replace(",", ".") if re.search(r",\d{1,2}$", s) else s.replace(",", "")
    elif re.search(r"\.\d{3}$", s) and not re.search(r"\.\d{2}$", s):
        s = s.replace(".", "")
    try:
        return float(s) * multiplier
    except ValueError:
        return np.nan


def parse_fbref_keepers(html_path):
    """Extrai goleiros da tabela stats_keeper do HTML salvo do FBref."""
    with open(html_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
    table = soup.find("table", id="stats_keeper")
    if table is None:
        return pd.DataFrame(columns=["nome", "clube", "minutos_gk"])
    rows = []
    tbody = table.find("tbody")
    if tbody is None:
        return pd.DataFrame(columns=["nome", "clube", "minutos_gk"])
    for tr in tbody.find_all("tr"):
        if tr.find("th", class_="thead"):
            continue
        player_td = tr.find("td", {"data-stat": "player"})
        if not player_td:
            continue
        link = player_td.find("a")
        nome = link.get_text(strip=True) if link else player_td.get_text(strip=True)
        team_td = tr.find("td", {"data-stat": "team"})
        clube = team_td.get_text(strip=True) if team_td else ""
        min_td = tr.find("td", {"data-stat": "gk_minutes"})
        minutos = min_td.get_text(strip=True).replace(",", "") if min_td else "0"
        rows.append({"nome": nome, "clube": clube, "minutos_gk": minutos})
    return pd.DataFrame(rows)


---
## Etapa 1 — Construção da Base de Dados [20%]


In [ ]:
# @title 1.1 Carregar FBref (V2) e mapear schema do enunciado

# V2 é superset do CSV base; V1 carregado para comparação de cobertura
df_v1 = pd.read_csv(BASE_DIR / "fbref_big5_brasileirao_2526.csv")
df_raw = pd.read_csv(BASE_DIR / "fbref_big5_brasileirao_2526_V2.csv")

# Renomear coluna problemática 1/3
if "1/3" in df_raw.columns:
    df_raw = df_raw.rename(columns={"1/3": "passes_ultimo_terco"})

print(f"FBref V1: {df_v1.shape[0]} jogadores, {df_v1.shape[1]} colunas")
print(f"FBref V2: {df_raw.shape[0]} jogadores, {df_raw.shape[1]} colunas")
print(f"Colunas extras na V2: {len(set(df_raw.columns) - set(df_v1.columns))}")

# Derivar chutes totais (Sh/90 * 90s ou FotMob)
if "Sh/90" in df_raw.columns and "90s" in df_raw.columns:
    chutes_calc = pd.to_numeric(df_raw["Sh/90"], errors="coerce") * pd.to_numeric(df_raw["90s"], errors="coerce")
elif "Chutes_por_90" in df_raw.columns and "Min" in df_raw.columns:
    chutes_calc = pd.to_numeric(df_raw["Chutes_por_90"], errors="coerce") * pd.to_numeric(df_raw["Min"], errors="coerce") / 90
else:
    chutes_calc = np.nan

df = pd.DataFrame({
    "nome": df_raw["Player"],
    "posicao_raw": df_raw["Pos"],
    "posicao": df_raw["Pos"].apply(simplificar_posicao),
    "idade": df_raw["Age"].apply(parse_idade),
    "nacionalidade": df_raw["Nation"],
    "clube": df_raw["Squad"],
    "liga": df_raw["liga"],
    "jogos": pd.to_numeric(df_raw["MP"], errors="coerce"),
    "minutos": pd.to_numeric(df_raw["Min"], errors="coerce"),
    "gols": pd.to_numeric(df_raw["Gls"], errors="coerce").fillna(0),
    "assistencias": pd.to_numeric(df_raw["Ast"], errors="coerce").fillna(0),
    "amarelos": pd.to_numeric(df_raw["CrdY"], errors="coerce").fillna(0),
    "vermelhos": pd.to_numeric(df_raw["CrdR"], errors="coerce").fillna(0),
    "passes_tent": pd.to_numeric(df_raw["Att"], errors="coerce"),
    "pct_passes": pd.to_numeric(df_raw["Cmp%"], errors="coerce"),
    "passes_prog": pd.to_numeric(df_raw.get("passes_ultimo_terco", np.nan), errors="coerce"),
    "chutes": chutes_calc,
    "chutes_gol": pd.to_numeric(df_raw["SoT"], errors="coerce"),
    "xG": pd.to_numeric(df_raw.get("xG", np.nan), errors="coerce"),
    "desarmes": pd.to_numeric(df_raw["Tkl"], errors="coerce"),
    "interceptacoes": pd.to_numeric(df_raw["Int"], errors="coerce"),
    "bloqueios": pd.to_numeric(df_raw.get("blocks", np.nan), errors="coerce"),
    "cortes": pd.to_numeric(df_raw["Clr"], errors="coerce"),
    "defesas_gk": pd.to_numeric(df_raw["Saves"], errors="coerce"),
    "pct_defesas": pd.to_numeric(df_raw["Save%"], errors="coerce"),
    "clean_sheets": pd.to_numeric(df_raw["CS"], errors="coerce"),
    "avaliacao_fotmob": pd.to_numeric(df_raw.get("Avaliacao_FotMob", np.nan), errors="coerce"),
    "90s": pd.to_numeric(df_raw["90s"], errors="coerce"),
    "starts": pd.to_numeric(df_raw["Starts"], errors="coerce"),
})

df["nome_norm"] = df["nome"].apply(normalizar_nome)
df.head(3)


In [ ]:
# @title 1.2 Raspagem Transfermarkt (BeautifulSoup + requests)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
}

BASE_TM_URL = (
    "https://www.transfermarkt.com.br/spieler-statistik/wertvollstespieler/"
    "marktwertetop/plus/0"
)
N_PAGES = 8
TM_RAW_PATH = DATA_DIR / "transfermarkt_raw.csv"


def scrape_transfermarkt(n_pages=N_PAGES):
    rows = []
    for page in range(1, n_pages + 1):
        url = f"{BASE_TM_URL}?galerie=0&page={page}"
        print(f"Raspando página {page}/{n_pages}...")
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.content, "html.parser")
        table = soup.find("table", class_="items")
        if table is None:
            print(f"  Tabela não encontrada na página {page}")
            continue
        tbody = table.find("tbody")
        if tbody is None:
            continue
        for tr in tbody.find_all("tr"):
            classes = tr.get("class", [])
            if "odd" not in classes and "even" not in classes:
                continue
            name_td = tr.find("td", class_="hauptlink")
            if not name_td:
                continue
            link = name_td.find("a")
            if not link:
                continue
            nome = link.get_text(strip=True)
            valor_tds = tr.find_all("td", class_="rechts hauptlink")
            valor_str = valor_tds[-1].get_text(strip=True) if valor_tds else ""
            pos_td = tr.find("td", class_=lambda c: c and "posrela" in c)
            posicao_tm = pos_td.get_text(strip=True) if pos_td else ""
            rows.append({
                "nome": nome,
                "posicao_tm": posicao_tm,
                "valor_mercado_str": valor_str,
            })
        if page < n_pages:
            time.sleep(2)
    return pd.DataFrame(rows)


if TM_RAW_PATH.exists():
    print(f"Carregando raspagem existente: {TM_RAW_PATH}")
    df_tm = pd.read_csv(TM_RAW_PATH)
else:
    try:
        df_tm = scrape_transfermarkt(N_PAGES)
        df_tm.to_csv(TM_RAW_PATH, index=False)
        print(f"Raspagem salva em {TM_RAW_PATH}")
    except Exception as exc:
        print(f"Erro na raspagem: {exc}")
        print("Execute novamente ou faça upload de transfermarkt_raw.csv em dados/")
        df_tm = pd.DataFrame(columns=["nome", "posicao_tm", "valor_mercado_str"])

df_tm["valor_mercado"] = df_tm["valor_mercado_str"].apply(parse_valor_mercado)
df_tm["nome_norm"] = df_tm["nome"].apply(normalizar_nome)
# Um jogador pode aparecer em páginas diferentes — manter maior valor
df_tm = (
    df_tm.sort_values("valor_mercado", ascending=False)
    .drop_duplicates(subset="nome_norm", keep="first")
    .reset_index(drop=True)
)
print(f"Jogadores Transfermarkt (únicos): {len(df_tm)}")
df_tm.head()


In [ ]:
# @title 1.3 Merge left (FBref base) + estatísticas do merge

df = df.merge(
    df_tm[["nome_norm", "nome", "valor_mercado"]].rename(columns={"nome": "nome_tm"}),
    on="nome_norm",
    how="left",
)

n_fbref = len(df)
n_com_valor = df["valor_mercado"].notna().sum()
cobertura = 100 * n_com_valor / n_fbref

print("=== Estatísticas do merge ===")
print(f"Jogadores FBref (base):     {n_fbref}")
print(f"Jogadores Transfermarkt:    {len(df_tm)}")
print(f"Com valor de mercado:       {n_com_valor} ({cobertura:.1f}%)")
print(f"Sem valor (NaN):            {n_fbref - n_com_valor}")

df_final_path = DATA_DIR / "dataset_final.csv"
df.to_csv(df_final_path, index=False)
print(f"\nDataset final salvo: {df_final_path}")
df[["nome", "clube", "liga", "posicao", "minutos", "valor_mercado"]].head(10)


In [ ]:
# @title 1.4 Validação com HTML local (goleiros FBref)

html_bra = BASE_DIR / "data_html" / "bra_keepers.html"
html_big5 = BASE_DIR / "data_html" / "big5_keepers.html"

keepers_bra = parse_fbref_keepers(html_bra) if html_bra.exists() else pd.DataFrame()
keepers_big5 = parse_fbref_keepers(html_big5) if html_big5.exists() else pd.DataFrame()

gk_csv_bra = df[(df["liga"] == "Brasileirao") & (df["posicao"] == "GK")].shape[0]
gk_csv_big5 = df[(df["liga"] == "Big5") & (df["posicao"] == "GK")].shape[0]

print("=== Validação HTML vs CSV ===")
print(f"Goleiros Brasileirão no CSV:  {gk_csv_bra}")
print(f"Goleiros Brasileirão no HTML: {len(keepers_bra)}")
print(f"Goleiros Big5 no CSV:         {gk_csv_big5}")
print(f"Goleiros Big5 no HTML:        {len(keepers_big5)}")
print("\nO HTML confirma a cobertura de goleiros; o CSV V2 é a fonte principal.")
if len(keepers_bra) > 0:
    display(keepers_bra.head(5))


---
## Etapa 2 — Pré-processamento e EDA [20%]


In [ ]:
# @title 2.1 Limpeza, imputação posicional e scaler

# Colunas numéricas do dataset
num_cols = [
    "jogos", "minutos", "gols", "assistencias", "amarelos", "vermelhos",
    "passes_tent", "pct_passes", "passes_prog", "chutes", "chutes_gol", "xG",
    "desarmes", "interceptacoes", "bloqueios", "cortes",
    "defesas_gk", "pct_defesas", "clean_sheets", "idade", "avaliacao_fotmob",
]

for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Imputar 0 em stats posicionais (NaN = não aplicável à posição)
pos_cols_zero = [
    "chutes", "chutes_gol", "xG", "desarmes", "interceptacoes",
    "bloqueios", "cortes", "defesas_gk", "pct_defesas", "clean_sheets",
    "passes_prog", "passes_tent", "pct_passes",
]
for col in pos_cols_zero:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Mediana por posição para métricas de eficiência (jogadores sem FotMob)
for col in ["xG", "avaliacao_fotmob"]:
    if col in df.columns:
        medianas = df.groupby("posicao")[col].transform("median")
        df[col] = df[col].fillna(medianas).fillna(0)

# Extrair código de país da nacionalidade (ex: br BRA -> BRA)
df["pais"] = df["nacionalidade"].astype(str).str.extract(r"([A-Z]{3})\s*$", expand=False)
df["pais"] = df["pais"].fillna(df["nacionalidade"].astype(str).str[-3:])

print("Distribuição por posição:")
print(df["posicao"].value_counts())
print("\nDistribuição por liga:")
print(df["liga"].value_counts())


In [ ]:
# @title 2.2 Visualização 1 — Distribuição do valor de mercado

fig, ax = plt.subplots(figsize=(10, 5))
valores = df["valor_mercado"].dropna()
mediana_valor = valores.median()
ax.hist(valores / 1e6, bins=30, color="steelblue", edgecolor="white", alpha=0.85)
ax.axvline(mediana_valor / 1e6, color="crimson", linestyle="--", linewidth=2,
           label=f"Mediana: € {mediana_valor/1e6:.1f} Mi")
ax.set_title("Distribuição do Valor de Mercado (Transfermarkt)")
ax.set_xlabel("Valor de mercado (milhões de euros)")
ax.set_ylabel("Número de jogadores")
ax.legend()
plt.tight_layout()
plt.show()

print("""
Interpretação: apenas ~{:.0f}% dos jogadores do FBref possuem valor de mercado no
Transfermarkt (top ~500 mundial). A mediana reflete o mercado de elite — jogadores
abaixo dessa faixa podem representar oportunidades subvalorizadas para a federação.
""".format(cobertura))


In [ ]:
# @title 2.3 Visualização 2 — Perfil médio por posição

metricas_pos = ["gols", "assistencias", "desarmes", "passes_prog", "xG"]
perfil = df.groupby("posicao")[metricas_pos].mean().reindex(["GK", "DF", "MF", "FW"])

fig, ax = plt.subplots(figsize=(12, 6))
perfil.plot(kind="bar", ax=ax, width=0.8)
ax.set_title("Perfil Médio por Posição — Métricas-Chave")
ax.set_xlabel("Posição")
ax.set_ylabel("Média na temporada")
ax.legend(title="Métrica", bbox_to_anchor=(1.02, 1))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("""
Interpretação: atacantes concentram gols e xG; defensores e meias lideram em
desarmes; meias se destacam em passes progressivos. Essa separação natural valida
a estratégia de preencher zeros posicionais antes da clusterização.
""")
display(perfil.round(2))


In [ ]:
# @title 2.4 Visualização 3 — Top-10 nacionalidades por valor médio

MIN_JOGADORES = 5
nat_stats = (
    df.dropna(subset=["valor_mercado"])
    .groupby("pais")
    .agg(valor_medio=("valor_mercado", "mean"), n=("nome", "count"))
    .query(f"n >= {MIN_JOGADORES}")
    .sort_values("valor_medio", ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(nat_stats.index[::-1], nat_stats["valor_medio"][::-1] / 1e6, color="teal")
ax.set_title(f"Top-10 Nacionalidades por Valor Médio (mín. {MIN_JOGADORES} jogadores)")
ax.set_xlabel("Valor médio (milhões de euros)")
ax.set_ylabel("Nacionalidade")
plt.tight_layout()
plt.show()

print("""
Interpretação: nacionalidades com maior valor médio concentram talentos de elite
nas Big 5. Para a Copa 2026 (48 seleções), federações de mercados menores podem
buscar jogadores de países com bom custo-benefício estatístico.
""")
display((nat_stats / 1e6).round(2).rename(columns={"valor_medio": "valor_medio_Mi"}))


In [ ]:
# @title 2.5 Visualização 4 — Minutos por liga

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x="liga", y="minutos", ax=ax, palette=["#4C72B0", "#DD8452"])
ax.set_title("Distribuição de Minutos Jogados por Liga")
ax.set_xlabel("Liga")
ax.set_ylabel("Minutos na temporada")
plt.tight_layout()
plt.show()

print("""
Interpretação: Big5 e Brasileirão têm perfis de minutos comparáveis na mediana,
mas o Brasileirão pode ter mais valores ausentes em stats avançadas (cobertura Opta).
Isso deve ser considerado ao interpretar clusters com jogadores brasileiros.
""")


In [ ]:
# @title 2.6 Visualização 5 — Heatmap de correlação (features de clusterização)

cluster_preview_cols = [
    "minutos", "gols", "assistencias", "xG", "passes_prog", "pct_passes",
    "desarmes", "interceptacoes", "cortes", "chutes", "chutes_gol",
    "defesas_gk", "clean_sheets", "idade",
]
corr = df[cluster_preview_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax, square=True)
ax.set_title("Correlação entre Features de Clusterização")
plt.tight_layout()
plt.show()

print("""
Interpretação: gols e xG correlacionam fortemente; defesas de goleiro são
ortogonais a métricas ofensivas — o que ajuda o K-Means a separar perfis.
""")


---
## Etapa 3 — Clusterização [25%]


In [ ]:
# @title 3.1 Preparar features e escolher k (Elbow + Silhouette)

le_pos = LabelEncoder()
df["posicao_enc"] = le_pos.fit_transform(df["posicao"])

cluster_features = [
    "minutos", "gols", "assistencias", "xG", "passes_prog", "pct_passes",
    "desarmes", "interceptacoes", "cortes", "chutes", "chutes_gol",
    "defesas_gk", "clean_sheets", "idade", "posicao_enc",
]

# Handle remaining NaNs in cluster features (specifically 'minutos' and 'idade')
# 'gols' and 'assistencias' were already filled in 1.1
# 'xG' and 'avaliacao_fotmob' were filled in 2.1
# Other positional stats in pos_cols_zero were filled in 2.1
for col in ["minutos", "idade"]:
    if col in df.columns:
        if col == "idade":
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(0)

X_cluster = df[cluster_features].copy()
scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

K_RANGE = range(2, 11)
inertias = []
silhouettes = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_cluster_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_cluster_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(K_RANGE), inertias, "o-", color="steelblue")
axes[0].set_title("Método do Cotovelo (Inércia)")
axes[0].set_xlabel("Número de clusters (k)")
axes[0].set_ylabel("Inércia")

axes[1].plot(list(K_RANGE), silhouettes, "o-", color="darkorange")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("Número de clusters (k)")
axes[1].set_ylabel("Silhouette")

plt.tight_layout()
plt.show()

# Escolha automática: maior silhouette (ajuste manual se necessário)
K_OPTIMAL = list(K_RANGE)[int(np.argmax(silhouettes))]
print(f"k sugerido pelo maior Silhouette Score: {K_OPTIMAL}")
print("Ajuste K_OPTIMAL manualmente se o cotovelo indicar outro valor.")


In [ ]:
# @title 3.2 K-Means, PCA e caracterização dos clusters

# Use o k sugerido ou defina manualmente
K_OPTIMAL = K_OPTIMAL if "K_OPTIMAL" in dir() else 4

kmeans = KMeans(n_clusters=K_OPTIMAL, random_state=RANDOM_STATE, n_init=10)
df["cluster"] = kmeans.fit_predict(X_cluster_scaled)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_cluster_scaled)
df["pca1"] = X_pca[:, 0]
df["pca2"] = X_pca[:, 1]
var_exp = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(df["pca1"], df["pca2"], c=df["cluster"], cmap="tab10", alpha=0.6, s=30)
ax.set_title("Clusters de Jogadores (PCA 2D)")
ax.set_xlabel(f"PC1 ({var_exp[0]*100:.1f}% variância)")
ax.set_ylabel(f"PC2 ({var_exp[1]*100:.1f}% variância)")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.show()

# Tabela-resumo por cluster
resumo_cols = [
    "gols", "assistencias", "xG", "desarmes", "interceptacoes", "cortes",
    "defesas_gk", "clean_sheets", "minutos", "idade", "valor_mercado",
]
cluster_resumo = df.groupby("cluster")[resumo_cols].mean().round(2)
cluster_pos = df.groupby("cluster")["posicao"].agg(lambda x: x.value_counts().index[0])
cluster_resumo["posicao_dominante"] = cluster_pos
cluster_resumo["n_jogadores"] = df.groupby("cluster").size()

# Nomear clusters automaticamente
def nomear_cluster(row):
    if row.get("defesas_gk", 0) > 20 and row.get("gols", 0) < 2:
        return "Goleiros"
    if row.get("gols", 0) > 3 or row.get("xG", 0) > 2:
        return "Atacantes"
    if row.get("desarmes", 0) > 30 or row.get("cortes", 0) > 40:
        return "Defensores"
    if row.get("passes_prog", 0) > 15 or row.get("assistencias", 0) > 2:
        return "Meias"
    return f"Perfil {int(row.name)}"

cluster_resumo["nome_cluster"] = cluster_resumo.apply(nomear_cluster, axis=1)
df["nome_cluster"] = df["cluster"].map(cluster_resumo["nome_cluster"])

print("=== Tabela-resumo por cluster ===")
display(cluster_resumo)

cluster_resumo.to_csv(DATA_DIR / "clusters_resumo.csv")
print(f"Salvo em {DATA_DIR / 'clusters_resumo.csv'}")


---
## Etapa 4 — Classificação [25%]


In [ ]:
# @title 4.1 Target binário e preparação de features (sem leakage)

mediana_min = df["minutos"].median()
df["target"] = (df["minutos"] > mediana_min).astype(int)

print(f"Mediana de minutos: {mediana_min:.0f}")
print("Distribuição do target:")
print(df["target"].value_counts(normalize=True).rename({0: "Rotação/Reserva", 1: "Titular Regular"}))

le_liga = LabelEncoder()
df["liga_enc"] = le_liga.fit_transform(df["liga"].astype(str))

# Features para classificação — SEM minutos, 90s, starts, valor_mercado
feature_cols_clf = [
    "gols", "assistencias", "xG", "passes_prog", "pct_passes",
    "desarmes", "interceptacoes", "bloqueios", "cortes",
    "chutes", "chutes_gol", "defesas_gk", "pct_defesas", "clean_sheets",
    "idade", "amarelos", "vermelhos", "jogos",
    "posicao_enc", "liga_enc", "avaliacao_fotmob",
]

X_clf = df[feature_cols_clf].fillna(0)
y_clf = df["target"]

scaler_clf = StandardScaler()
X_clf_scaled = scaler_clf.fit_transform(X_clf)

print(f"\nFeatures: {len(feature_cols_clf)} | Amostras: {len(X_clf)}")


In [ ]:
# @title 4.2 Modelos com validação cruzada estratificada (k=5)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["accuracy", "precision", "recall", "f1"]

modelos = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Regressão Logística": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE
    ),
}

resultados = []
for nome, modelo in modelos.items():
    scores = cross_validate(modelo, X_clf_scaled, y_clf, cv=cv, scoring=scoring)
    resultados.append({
        "Modelo": nome,
        "Acurácia": scores["test_accuracy"].mean(),
        "Precisão": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1-Score": scores["test_f1"].mean(),
    })

df_metricas = pd.DataFrame(resultados)
df_metricas[["Acurácia", "Precisão", "Recall", "F1-Score"]] = (
    df_metricas[["Acurácia", "Precisão", "Recall", "F1-Score"]].round(4)
)
print("=== Métricas (média CV 5-fold) ===")
display(df_metricas)

df_metricas.to_csv(DATA_DIR / "metricas_classificacao.csv", index=False)


In [ ]:
# @title 4.3 Matrizes de confusão e importância de features

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (nome, modelo) in zip(axes, modelos.items()):
    modelo.fit(X_clf_scaled, y_clf)
    y_pred = modelo.predict(X_clf_scaled)
    cm = confusion_matrix(y_clf, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Rotação (0)", "Titular (1)"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Matriz de Confusão — {nome}")

plt.tight_layout()
plt.show()

# Feature importance — Random Forest
rf = modelos["Random Forest"]
rf.fit(X_clf_scaled, y_clf)
imp_rf = pd.Series(rf.feature_importances_, index=feature_cols_clf).sort_values(ascending=False)

lr = modelos["Regressão Logística"]
lr.fit(X_clf_scaled, y_clf)
imp_lr = pd.Series(np.abs(lr.coef_[0]), index=feature_cols_clf).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
imp_rf.head(10).plot(kind="barh", ax=axes[0], color="forestgreen")
axes[0].set_title("Top 10 — Random Forest")
imp_lr.head(10).plot(kind="barh", ax=axes[1], color="royalblue")
axes[1].set_title("Top 10 — Regressão Logística")
plt.tight_layout()
plt.show()

print("""
Reflexão (Precisão vs Recall):
- Falso positivo (reserva classificado como titular): risco de convocar jogador despreparado.
- Falso negativo (titular não identificado): perder talento em forma.
Para a Copa, o Recall costuma ser mais crítico — não deixar de identificar titulares reais.
""")


---
## Etapa 5 — Insumos para Recomendação Estratégica [10%]


In [ ]:
# @title 5.1 Resumo para o relatório PDF

# Treinar RF final para predições
rf_final = RandomForestClassifier(
    n_estimators=100, class_weight="balanced", random_state=RANDOM_STATE
)
rf_final.fit(X_clf_scaled, y_clf)
df["pred_titular"] = rf_final.predict(X_clf_scaled)
df["prob_titular"] = rf_final.predict_proba(X_clf_scaled)[:, 1]

top_features = imp_rf.head(5).index.tolist()

print("=" * 60)
print("RESUMO PARA RELATÓRIO — ETAPA 5")
print("=" * 60)

print("\n--- Clusters e composição de elenco ---")
for c in sorted(df["cluster"].unique()):
    nome_c = cluster_resumo.loc[c, "nome_cluster"]
    n = cluster_resumo.loc[c, "n_jogadores"]
    pos = cluster_resumo.loc[c, "posicao_dominante"]
    vm = cluster_resumo.loc[c, "valor_mercado"]
    vm_str = f"€ {vm/1e6:.1f} Mi" if not pd.isna(vm) else "N/A"
    print(f"  Cluster {c} ({nome_c}): {n} jogadores | Posição dominante: {pos} | Valor médio: {vm_str}")

print("\n--- Jogadores Nível Copa (exemplos para citação) ---")
candidatos = (
    df[(df["pred_titular"] == 1) & (df["prob_titular"] > 0.7)]
    .nlargest(10, "prob_titular")[
        ["nome", "clube", "liga", "posicao", "minutos", "gols", "assistencias", "xG", "nome_cluster"]
    ]
)
display(candidatos.head(5))

print("\n--- Limitações da abordagem ---")
limitacoes = [
    "Target proxy (minutos) não captura qualidade tática ou momento de forma.",
    "Merge Transfermarkt cobre apenas ~{:.0f}% dos jogadores.".format(cobertura),
    "Ausência de dados de lesões, química de elenco e contexto de partida.",
    "Scouting profissional usa vídeo, relatórios subjetivos e testes físicos.",
    "Cobertura Opta menor no Brasileirão pode enviesar comparações com Big5.",
]
for i, lim in enumerate(limitacoes, 1):
    print(f"  {i}. {lim}")

# Export final
export_cols = [
    "nome", "clube", "liga", "posicao", "minutos", "valor_mercado",
    "cluster", "nome_cluster", "target", "pred_titular", "prob_titular",
] + feature_cols_clf
df[export_cols].to_csv(DATA_DIR / "dataset_completo_analise.csv", index=False)

print(f"\nArquivos exportados em {DATA_DIR.resolve()}:")
for f in sorted(DATA_DIR.glob("*.csv")):
    print(f"  - {f.name}")
print("\n✓ Pipeline concluído. Use os outputs acima para montar o relatório PDF.")
